# 🔍 Notebook 01 : Exploration des Données - Détection de Fraude

**Auteur** : Oumaro Titans DJIGUIMDE  
**Date** : Février 2026  
**Objectif** : Explorer et comprendre les patterns de fraude dans les transactions Mobile Money

---

## 🎯 Objectifs de ce notebook

1. Charger et inspecter le dataset de transactions
2. Analyser le déséquilibre des classes (fraude vs normal)
3. Identifier les patterns de fraude
4. Analyser les distributions par variables
5. Visualiser les tendances temporelles
6. Comprendre les corrélations avec la fraude

## 📦 Importation des bibliothèques

In [ ]:
# Manipulation de données
import pandas as pd
import numpy as np

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Utilitaires
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuration
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# Style des graphiques
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('Set2')
plt.rcParams['figure.figsize'] = (12, 6)

print("✅ Bibliothèques importées avec succès")

## 1️⃣ Chargement des données

In [ ]:
# Chargement du dataset
df = pd.read_csv('../data/transactions.csv', parse_dates=['date'])

print("="*70)
print("🚨 DATASET DE DÉTECTION DE FRAUDE CHARGÉ")
print("="*70)
print(f"📦 Nombre de transactions : {len(df):,}")
print(f"📋 Nombre de colonnes : {len(df.columns)}")
print(f"📅 Période : {df['date'].min().strftime('%Y-%m-%d')} → {df['date'].max().strftime('%Y-%m-%d')}")
print(f"\n🚨 Transactions frauduleuses : {df['is_fraud'].sum():,} ({df['is_fraud'].mean()*100:.2f}%)")
print(f"✅ Transactions normales : {(~df['is_fraud'].astype(bool)).sum():,} ({(~df['is_fraud'].astype(bool)).mean()*100:.2f}%)")
print("="*70)

## 2️⃣ Inspection du dataset

In [ ]:
# Aperçu des premières lignes
print("🔍 Aperçu des 10 premières transactions :")
df.head(10)

In [ ]:
# Informations sur le dataset
print("📋 Informations sur le dataset :")
df.info()

In [ ]:
# Statistiques descriptives
print("📊 Statistiques descriptives :")
df.describe()

In [ ]:
# Valeurs manquantes
print("⚠️  Valeurs manquantes :")
missing = df.isnull().sum()
if missing.sum() == 0:
    print("   ✅ Aucune valeur manquante !")
else:
    print(missing[missing > 0])

In [ ]:
# Doublons
print("🔄 Vérification des doublons :")
doublons = df.duplicated().sum()
print(f"   Nombre de doublons : {doublons}")

## 3️⃣ Analyse du déséquilibre des classes

**Point critique** : Les datasets de fraude sont toujours déséquilibrés car les fraudes sont rares.

In [ ]:
# Distribution fraude vs normal
fraud_counts = df['is_fraud'].value_counts()
fraud_pct = df['is_fraud'].value_counts(normalize=True) * 100

print("="*70)
print("🚨 DÉSÉQUILIBRE DES CLASSES")
print("="*70)
print(f"\nNormal (0) : {fraud_counts[0]:,} transactions ({fraud_pct[0]:.2f}%)")
print(f"Fraude (1) : {fraud_counts[1]:,} transactions ({fraud_pct[1]:.2f}%)")
print(f"\nRatio déséquilibre : 1:{fraud_counts[0]//fraud_counts[1]}")
print("="*70)

In [ ]:
# Visualisation du déséquilibre
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique en barres
fraud_counts.plot(kind='bar', ax=axes[0], color=['lightgreen', 'salmon'])
axes[0].set_title('Distribution des Classes', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Classe')
axes[0].set_ylabel('Nombre de transactions')
axes[0].set_xticklabels(['Normal', 'Fraude'], rotation=0)
axes[0].grid(True, alpha=0.3, axis='y')

# Pie chart
colors = ['lightgreen', 'salmon']
axes[1].pie(fraud_counts, labels=['Normal', 'Fraude'], autopct='%1.2f%%', 
            colors=colors, startangle=90, textprops={'fontsize': 12})
axes[1].set_title('Proportion Fraude vs Normal', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

## 4️⃣ Analyse des types de fraude

In [ ]:
# Distribution des types de fraude
fraud_types = df[df['is_fraud'] == 1]['fraud_type'].value_counts()
fraud_types_pct = df[df['is_fraud'] == 1]['fraud_type'].value_counts(normalize=True) * 100

print("="*70)
print("🚨 TYPES DE FRAUDES DÉTECTÉES")
print("="*70)
for fraud_type, count in fraud_types.items():
    pct = fraud_types_pct[fraud_type]
    print(f"{fraud_type:25s} : {count:5,} ({pct:5.1f}%)")
print("="*70)

In [ ]:
# Visualisation des types de fraude
plt.figure(figsize=(12, 6))
fraud_types.plot(kind='barh', color='coral')
plt.title('Distribution des Types de Fraude', fontsize=16, fontweight='bold')
plt.xlabel('Nombre de cas', fontsize=12)
plt.ylabel('Type de fraude', fontsize=12)
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 5️⃣ Analyse des montants

In [ ]:
# Comparaison montants fraude vs normal
print("="*70)
print("💰 ANALYSE DES MONTANTS")
print("="*70)

print("\n📊 Transactions NORMALES :")
normal_amounts = df[df['is_fraud'] == 0]['amount']
print(f"   Moyenne : {normal_amounts.mean():,.0f} FCFA")
print(f"   Médiane : {normal_amounts.median():,.0f} FCFA")
print(f"   Min : {normal_amounts.min():,.0f} FCFA")
print(f"   Max : {normal_amounts.max():,.0f} FCFA")

print("\n🚨 Transactions FRAUDULEUSES :")
fraud_amounts = df[df['is_fraud'] == 1]['amount']
print(f"   Moyenne : {fraud_amounts.mean():,.0f} FCFA")
print(f"   Médiane : {fraud_amounts.median():,.0f} FCFA")
print(f"   Min : {fraud_amounts.min():,.0f} FCFA")
print(f"   Max : {fraud_amounts.max():,.0f} FCFA")

print(f"\n📈 Différence moyenne : {fraud_amounts.mean() - normal_amounts.mean():,.0f} FCFA")
print(f"   Ratio : {fraud_amounts.mean() / normal_amounts.mean():.2f}x")
print("="*70)

In [ ]:
# Visualisation des distributions de montants
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogrammes
axes[0].hist(normal_amounts, bins=50, alpha=0.7, label='Normal', color='lightgreen', edgecolor='black')
axes[0].hist(fraud_amounts, bins=50, alpha=0.7, label='Fraude', color='salmon', edgecolor='black')
axes[0].set_title('Distribution des Montants', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Montant (FCFA)')
axes[0].set_ylabel('Fréquence')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Boxplots
data_to_plot = [normal_amounts, fraud_amounts]
axes[1].boxplot(data_to_plot, labels=['Normal', 'Fraude'], patch_artist=True,
                boxprops=dict(facecolor='lightblue'))
axes[1].set_title('Boxplot des Montants', fontsize=14, fontweight='bold')
axes[1].set_ylabel('Montant (FCFA)')
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 6️⃣ Analyse temporelle

In [ ]:
# Distribution des fraudes par heure
print("="*70)
print("🕐 ANALYSE HORAIRE DES FRAUDES")
print("="*70)

hourly_fraud = df.groupby(['hour', 'is_fraud']).size().unstack(fill_value=0)
hourly_fraud_rate = (hourly_fraud[1] / (hourly_fraud[0] + hourly_fraud[1]) * 100)

print("\nTaux de fraude par heure :")
print(hourly_fraud_rate.sort_values(ascending=False).head(10))

In [ ]:
# Visualisation par heure
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

# Nombre de transactions par heure
hourly_fraud.plot(kind='bar', stacked=False, ax=axes[0], color=['lightgreen', 'salmon'])
axes[0].set_title('Distribution des Transactions par Heure', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Heure')
axes[0].set_ylabel('Nombre de transactions')
axes[0].legend(['Normal', 'Fraude'])
axes[0].grid(True, alpha=0.3, axis='y')

# Taux de fraude par heure
hourly_fraud_rate.plot(kind='line', marker='o', ax=axes[1], color='red', linewidth=2, markersize=6)
axes[1].set_title('Taux de Fraude par Heure', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Heure')
axes[1].set_ylabel('Taux de fraude (%)')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=df['is_fraud'].mean()*100, color='orange', linestyle='--', label='Taux moyen')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Heures les plus risquées
heures_risque = hourly_fraud_rate[hourly_fraud_rate > df['is_fraud'].mean()*100]

print("\n⚠️  Heures à risque élevé (taux > moyenne) :")
for heure, taux in heures_risque.items():
    print(f"   {heure:02d}h : {taux:.2f}%")

## 7️⃣ Analyse par type de transaction

In [ ]:
# Fraude par type de transaction
trans_fraud = df.groupby(['transaction_type', 'is_fraud']).size().unstack(fill_value=0)
trans_fraud_rate = (trans_fraud[1] / (trans_fraud[0] + trans_fraud[1]) * 100)

print("="*70)
print("💳 FRAUDE PAR TYPE DE TRANSACTION")
print("="*70)
for trans_type, taux in trans_fraud_rate.sort_values(ascending=False).items():
    total = trans_fraud.loc[trans_type].sum()
    fraudes = trans_fraud.loc[trans_type, 1]
    print(f"{trans_type:15s} : {taux:5.2f}% ({fraudes:,} / {total:,})")
print("="*70)

In [ ]:
# Visualisation
plt.figure(figsize=(10, 6))
trans_fraud_rate.sort_values(ascending=True).plot(kind='barh', color='coral')
plt.title('Taux de Fraude par Type de Transaction', fontsize=16, fontweight='bold')
plt.xlabel('Taux de fraude (%)', fontsize=12)
plt.ylabel('Type de transaction', fontsize=12)
plt.axvline(x=df['is_fraud'].mean()*100, color='red', linestyle='--', label='Taux moyen')
plt.legend()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 8️⃣ Analyse par ville

In [ ]:
# Fraude par ville
city_fraud = df.groupby(['city', 'is_fraud']).size().unstack(fill_value=0)
city_fraud_rate = (city_fraud[1] / (city_fraud[0] + city_fraud[1]) * 100)

print("="*70)
print("🏙️  FRAUDE PAR VILLE")
print("="*70)
for city, taux in city_fraud_rate.sort_values(ascending=False).items():
    total = city_fraud.loc[city].sum()
    fraudes = city_fraud.loc[city, 1]
    print(f"{city:15s} : {taux:5.2f}% ({fraudes:,} / {total:,})")
print("="*70)

In [ ]:
# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Nombre de fraudes
city_fraud[1].sort_values(ascending=True).plot(kind='barh', ax=axes[0], color='salmon')
axes[0].set_title('Nombre de Fraudes par Ville', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Nombre de fraudes')
axes[0].grid(True, alpha=0.3, axis='x')

# Taux de fraude
city_fraud_rate.sort_values(ascending=True).plot(kind='barh', ax=axes[1], color='coral')
axes[1].set_title('Taux de Fraude par Ville', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Taux de fraude (%)')
axes[1].axvline(x=df['is_fraud'].mean()*100, color='red', linestyle='--', label='Taux moyen')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.show()

## 9️⃣ Analyse par opérateur

In [ ]:
# Fraude par opérateur
op_fraud = df.groupby(['operator', 'is_fraud']).size().unstack(fill_value=0)
op_fraud_rate = (op_fraud[1] / (op_fraud[0] + op_fraud[1]) * 100)

print("="*70)
print("📱 FRAUDE PAR OPÉRATEUR MOBILE MONEY")
print("="*70)
for operator, taux in op_fraud_rate.sort_values(ascending=False).items():
    total = op_fraud.loc[operator].sum()
    fraudes = op_fraud.loc[operator, 1]
    print(f"{operator:20s} : {taux:5.2f}% ({fraudes:,} / {total:,})")
print("="*70)

In [ ]:
# Visualisation
plt.figure(figsize=(10, 6))
op_fraud_rate.sort_values(ascending=True).plot(kind='barh', color='skyblue')
plt.title('Taux de Fraude par Opérateur Mobile Money', fontsize=16, fontweight='bold')
plt.xlabel('Taux de fraude (%)', fontsize=12)
plt.ylabel('Opérateur', fontsize=12)
plt.axvline(x=df['is_fraud'].mean()*100, color='red', linestyle='--', label='Taux moyen')
plt.legend()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 🔟 Analyse des features comportementales

In [ ]:
# Analyse nb_transactions_24h
print("="*70)
print("📊 ANALYSE DES FEATURES COMPORTEMENTALES")
print("="*70)

print("\n1️⃣ Nombre de transactions dans les 24h :")
print(f"   Normal : {df[df['is_fraud']==0]['nb_transactions_24h'].mean():.2f} (moyenne)")
print(f"   Fraude : {df[df['is_fraud']==1]['nb_transactions_24h'].mean():.2f} (moyenne)")

print("\n2️⃣ Changement de localisation :")
loc_fraud = df.groupby(['changement_localisation', 'is_fraud']).size().unstack(fill_value=0)
loc_fraud_rate = (loc_fraud[1] / (loc_fraud[0] + loc_fraud[1]) * 100)
print(f"   Pas de changement : {loc_fraud_rate[0]:.2f}% de fraude")
print(f"   Avec changement  : {loc_fraud_rate[1]:.2f}% de fraude")

print("\n3️⃣ Appareil différent :")
dev_fraud = df.groupby(['appareil_different', 'is_fraud']).size().unstack(fill_value=0)
dev_fraud_rate = (dev_fraud[1] / (dev_fraud[0] + dev_fraud[1]) * 100)
print(f"   Même appareil     : {dev_fraud_rate[0]:.2f}% de fraude")
print(f"   Appareil différent : {dev_fraud_rate[1]:.2f}% de fraude")

print("="*70)

In [ ]:
# Visualisation des features comportementales
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# nb_transactions_24h
df.groupby('is_fraud')['nb_transactions_24h'].mean().plot(kind='bar', ax=axes[0], 
                                                            color=['lightgreen', 'salmon'])
axes[0].set_title('Nb Transactions 24h (moyenne)', fontsize=12, fontweight='bold')
axes[0].set_xticklabels(['Normal', 'Fraude'], rotation=0)
axes[0].set_ylabel('Moyenne')
axes[0].grid(True, alpha=0.3, axis='y')

# Changement localisation
loc_fraud_rate.plot(kind='bar', ax=axes[1], color=['lightgreen', 'salmon'])
axes[1].set_title('Taux Fraude - Changement Localisation', fontsize=12, fontweight='bold')
axes[1].set_xticklabels(['Non', 'Oui'], rotation=0)
axes[1].set_ylabel('Taux de fraude (%)')
axes[1].grid(True, alpha=0.3, axis='y')

# Appareil différent
dev_fraud_rate.plot(kind='bar', ax=axes[2], color=['lightgreen', 'salmon'])
axes[2].set_title('Taux Fraude - Appareil Différent', fontsize=12, fontweight='bold')
axes[2].set_xticklabels(['Non', 'Oui'], rotation=0)
axes[2].set_ylabel('Taux de fraude (%)')
axes[2].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 1️⃣1️⃣ Matrice de corrélation

In [ ]:
# Sélection des colonnes numériques
numeric_cols = ['hour', 'amount', 'nb_transactions_24h', 'changement_localisation', 
                'appareil_different', 'is_fraud']

# Matrice de corrélation
corr_matrix = df[numeric_cols].corr()

# Visualisation
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm', 
            center=0, square=True, linewidths=1)
plt.title('Matrice de Corrélation', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("\n🔍 Corrélations avec is_fraud :")
print(corr_matrix['is_fraud'].sort_values(ascending=False))

## 📌 Résumé des observations

### ✅ Points clés découverts

1. **Déséquilibre des classes** :
   - ~6% de fraudes (réaliste selon les études GSMA)
   - Ratio ~1:16 (normal vs fraude)
   - **Implication** : Besoin de techniques de rééquilibrage (SMOTE, sous-échantillonnage)

2. **Types de fraudes** :
   - **Phishing** domine (~60%) - conforme aux études
   - **SIM Swap** (~15%) - menace croissante
   - **Bypass Cash-In** (~12%) - problème agents

3. **Montants** :
   - Fraudes : **2.3x plus élevées** que transactions normales
   - Montant moyen fraude : ~90K FCFA
   - **Implication** : Le montant est un signal fort

4. **Patterns temporels** :
   - Pics de fraude la **nuit** (23h-5h)
   - Heures à risque identifiées
   - **Implication** : Heure = feature importante

5. **Type de transaction** :
   - **Withdrawal** = type le plus risqué
   - Transfer aussi suspect
   - **Implication** : Type de transaction = signal fort

6. **Features comportementales** :
   - **Changement localisation** : Signal très fort de fraude
   - **Appareil différent** : Corrélé avec fraude
   - **Nb transactions** : Fraudeurs = activité plus intense

### 🎯 Prochaines étapes

➡️ **Notebook 02** : Prétraitement & Feature Engineering
- Encodage des variables catégorielles
- Normalisation des montants
- Création de nouvelles features
- Gestion du déséquilibre (SMOTE)
- Split train/test stratifié

➡️ **Notebook 03** : Modélisation
- Régression Logistique (baseline)
- Random Forest
- XGBoost (optionnel)
- Optimisation hyperparamètres
- Évaluation (Recall, Precision, F1, AUC-ROC)
- Feature Importance